This notebook I will see if clip can identify insect crops correctly according to their family or even order

## Imports

In [3]:
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel

## Model initialization

In [4]:
# Load CLIP model
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

pytorch_model.bin:  10%|#         | 62.9M/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
# Load image
from pathlib import Path
# Candidate texts
texts = [ # test 1
    "a photo of a insect",
    "a photo of a raindrop",
    "a photo of a Diptera",
    "a photo of Blattodea",
    "a photo of a Neuroptera",
    "a photo of a Lepidopetra",
    "a photo of a Pterophoridae",
    "a photo of a Crambidea"
]
texts = [ # test 2
    "a photo of a insect",
    'a photo of a dirt',
    'a photo of a raindrop',
    'a blurry photo'
]

folder = Path("../data/clusters_output_15_include_cluster_id/Rejected_pictures")
i = 0
for image_path in folder.iterdir():

    image = Image.open(image_path)



    # Preprocess image and text
    inputs = processor(
        text=texts,
        images=image,
        return_tensors="pt",
        padding=True
    )

    # Run model
    with torch.no_grad():
        outputs = model(**inputs)

    # Similarity between image and each text
    logits_per_image = outputs.logits_per_image

    # Convert to probabilities
    probs = logits_per_image.softmax(dim=1)

    # Get best match
    best_index = probs.argmax().item()
    print(image_path)
    print("Best text:", texts[best_index])
    print("Probability:", probs[0][best_index].item())
    i += 1
    if i == 10:
        break

../data/clusters_output_15_include_cluster_id/Rejected_pictures/0.52_475_224_19_470.jpg
Best text: a photo of a insect
Probability: 0.9286041855812073
../data/clusters_output_15_include_cluster_id/Rejected_pictures/0.64_837_482_15_128.jpg
Best text: a photo of a insect
Probability: 0.6307527422904968
../data/clusters_output_15_include_cluster_id/Rejected_pictures/0.57_213_42_9_043.jpg
Best text: a photo of a insect
Probability: 0.616439700126648
../data/clusters_output_15_include_cluster_id/Rejected_pictures/0.35_838_312_19_661.jpg
Best text: a photo of a insect
Probability: 0.9811493158340454
../data/clusters_output_15_include_cluster_id/Rejected_pictures/0.58_958_113_17_097.jpg
Best text: a photo of a insect
Probability: 0.8775414228439331
../data/clusters_output_15_include_cluster_id/Rejected_pictures/0.67_232_51_19_178.jpg
Best text: a photo of a insect
Probability: 0.9915781021118164
../data/clusters_output_15_include_cluster_id/Rejected_pictures/0.64_1012_162_19_219.jpg
Best text

## Conclusion

CLIP model is not good on order/family level especially on lower quality photos, even the ones with higher quality that for dino clustered easily, clip classifies them as ones that are part of another cluster.

CLIP model can be set between detection and clustering layers to verify that the crop is an insect and not a piece of dirt, raindrop, shadow, branch, etc. It cannot be used for deep domain classification reliably.

### Issues

The problem with this approach is that rejected photos by entomologist students and their mentor is not due a linear relation.
- Not due to size or resolution of the pic, some where identifiable at a worse resolution than rejected ones
- Not due it not being a insect, most rejected were in fact insects for various reasons
    - crops did not include full insect
    - crops included multiple similar and different insects due to their proximity